In [ ]:
import pandas as pd

df = pd.read_csv(
    "products_details_tuan.csv",
    engine="python",
    sep=None,
    on_bad_lines="skip"
)

df.columns = (
    df.columns
    .astype(str)
    .str.replace("\ufeff", "", regex=False) 
    .str.strip()                            
)

print(f"Đã đọc xong file → {len(df)} dòng")

In [ ]:
# Tổng dòng + trùng link

# Tổng dòng ban đầu
total_rows = len(df)
print(f"Tổng số dòng trong file: {total_rows}")

# Đếm trùng link TRƯỚC khi xóa
dup_before = df['link'].duplicated().sum()
print(f"Số dòng bị trùng ở cột 'link': {dup_before}")


# Xóa trùng
df = df.drop_duplicates(subset='link', keep='first').copy()

# Kiểm tra sau khi xóa
dup_after = df['link'].duplicated().sum()
print(f"\nĐã xóa {dup_before} dòng trùng link")
print(f"Số dòng còn lại: {len(df)}")

In [ ]:
# Hàm kiểm tra missing 
def is_missing(series):
    return (
        series.isna() |
        (series.astype(str).str.strip() == "") |
        (series.astype(str).str.lower().isin(["nan", "null", "-", "reviews"]))
    )

In [ ]:
# Loại bỏ các dòng thiếu category + original_price + name

import re
import numpy as np

print("LOẠI BỎ DÒNG THIẾU category + original_price + name")

# Đếm trước
miss_cat = is_missing(df['category']).sum()
miss_price = is_missing(df['original_price']).sum()
miss_name = is_missing(df['name']).sum()

print(f"Thiếu category       : {miss_cat} dòng")
print(f"Thiếu original_price : {miss_price} dòng")
print(f"Thiếu name           : {miss_name} dòng")

# Mask loại bỏ
mask_remove = is_missing(df['category']) | is_missing(df['original_price']) | is_missing(df['name'])
n_removed = mask_remove.sum()
print(f"→ Tổng dòng bị loại  : {n_removed}")

# URL bị loại
urls_removed = df.loc[mask_remove, 'link'].astype(str).str.strip().dropna().tolist()

# Loại bỏ
df = df[~mask_remove].copy()
print(f"Số dòng còn lại      : {len(df)}")

In [ ]:
# CHUẨN HÓA CÁC CỘT: original_price, discount_percent, price, comment, rating, sold

print("CHUẨN HÓA CÁC CỘT GIÁ TRỊ")

# === Hàm hỗ trợ ===
def extract_number(text):
    if pd.isna(text):
        return np.nan
    s = str(text).strip()
    digits = re.sub(r"[^\d]", "", s)
    return int(digits) if digits else np.nan

def extract_sold(text):
    """Xử lý sold: 1.5K → 1500, 15.5K → 15500, 2M → 2000000"""
    if pd.isna(text):
        return 0
    s = str(text).strip().upper()
    if not s or s in ["NAN", "NULL", "-"]:
        return 0

    # Bỏ "ĐÃ BÁN", dấu phẩy, khoảng trắng
    s = re.sub(r"[^0-9KM\.]", "", s)

    # Match: 1.5K, 15.5K, 2K, 1.5M, etc.
    match = re.match(r"(\d+)\.(\d+)([KM])", s)
    if match:
        int_part, dec_part, unit = match.groups()
        value = int(int_part + dec_part)
        if unit == "K":
            return value * (100 if len(dec_part) == 1 else 10)
        elif unit == "M":
            return value * (100000 if len(dec_part) == 1 else 10000)
    
    # Match: 2K, 3M
    match = re.match(r"(\d+)([KM])", s)
    if match:
        num, unit = match.groups()
        if unit == "K":
            return int(num) * 1000
        elif unit == "M":
            return int(num) * 1000000

    # Chỉ số thường
    digits = re.sub(r"[^\d]", "", s)
    return int(digits) if digits else 0

# original_price: chỉ giữ số
if 'original_price' in df.columns:
    df['original_price'] = df['original_price'].apply(extract_number)
    print(f"original_price → chuẩn hóa xong (chỉ số)")
else:
    print("Cảnh báo: Không tìm thấy cột 'original_price'")

# discount_percent → giữ %, thiếu = 0%
if 'discount_percent' in df.columns:
    df['discount_percent'] = df['discount_percent'].astype(str).str.strip()
    df['discount_percent'] = df['discount_percent'].str.replace(r"[^\d%]", "", regex=True)
    df['discount_percent'] = df['discount_percent'].replace({"": "0%", "NAN": "0%", "NULL": "0%", "-": "0%"})
    print(f"discount_percent → chuẩn hóa xong (giữ %, thiếu = 0%)")
else:
    df['discount_percent'] = "0%"
    print(f"discount_percent → không có cột → tạo mới với giá trị mặc định 0%")

# price: chuẩn hóa số + tính lại từ original_price * discount_percent 
if 'price' in df.columns or 'price' not in df.columns:
    # Chuẩn hóa price hiện có
    if 'price' in df.columns:
        df['price'] = df['price'].apply(extract_number)
    else:
        df['price'] = np.nan

    # Tính lại price = original_price * (1 - discount%)
    def calc_price(row):
        orig = row['original_price']
        disc = row['discount_percent']
        if pd.isna(orig) or orig == 0:
            return np.nan
        if pd.isna(disc) or disc == "0%":
            return orig
        # Lấy % từ discount_percent
        try:
            disc_pct = int(re.sub(r"[^\d]", "", disc)) / 100
            return int(orig * (1 - disc_pct))
        except:
            return orig  # nếu lỗi, giữ nguyên gốc

    df['price'] = df.apply(calc_price, axis=1)
    df['price'] = df['price'].astype('Int64')  # hỗ trợ NaN
    print(f"price → chuẩn hóa + tính lại từ original_price x discount_percent")

# comment: chỉ giữ số, thiếu = 0 
comment_cols = [c for c in df.columns if 'comment' in c.lower()]
if comment_cols:
    col = comment_cols[0]
    df[col] = df[col].apply(lambda x: extract_number(x) if pd.notna(x) else 0)
    df[col] = df[col].fillna(0).astype(int)
    print(f"{col} → chuẩn hóa xong (chỉ số, thiếu = 0)")
else:
    df['comment_count'] = 0
    print("comment → không có cột → tạo mới = 0")

# rating: thiếu = 0 
rating_cols = [c for c in df.columns if 'rating' in c.lower() or 'star' in c.lower()]
if rating_cols:
    col = rating_cols[0]
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    print(f"{col} → chuẩn hóa xong (thiếu = 0)")
else:
    df['rating'] = 0
    print("rating → không có cột → tạo mới = 0")

# sold: xử lý K/M → số nguyên 
sold_cols = [c for c in df.columns if 'sold' in c.lower() or 'bán' in c.lower()]
if sold_cols:
    col = sold_cols[0]
    df[col] = df[col].apply(extract_sold)
    print(f"{col} → chuẩn hóa xong (1.5K = 1500, 15.5K = 15500)")
else:
    df['sold'] = 0
    print("sold → không có cột → tạo mới = 0")

# Xem mẫu 5 dòng
print("\nMẪU 5 DÒNG DỮ LIỆU SAU KHI CHUẨN HÓA:")
cols_to_show = ['name', 'original_price', 'discount_percent', 'price', 
                'comment_count' if 'comment_count' in df.columns else df.columns[df.columns.str.contains('comment', case=False)][0] if any('comment' in c.lower() for c in df.columns) else 'comment_count',
                'rating' if 'rating' in df.columns else df.columns[df.columns.str.contains('rating|star', case=False)][0] if any('rating' in c.lower() or 'star' in c.lower() for c in df.columns) else 'rating',
                'sold' if 'sold' in df.columns else df.columns[df.columns.str.contains('sold|bán', case=False)][0] if any('sold' in c.lower() or 'bán' in c.lower() for c in df.columns) else 'sold',
                'link']
cols_to_show = [c for c in cols_to_show if c in df.columns][:8]

with pd.option_context('display.max_colwidth', 50):
    print(df[cols_to_show].head(5).to_string(index=False))

In [ ]:
# # Cấu trúc file csv khác nên clean kiểu khác
# import pandas as pd
# import numpy as np
# import re

# df = pd.read_csv("products_details_an.csv", engine="python", on_bad_lines="skip")
# df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()

# print(f"Đã đọc: {len(df)} dòng")

# # XÓA DÒNG comment_count = "Reviews"
# mask_rev = df['comment_count'].astype(str).str.strip() == "Reviews"
# n_rev = mask_rev.sum()
# df = df[~mask_rev].copy()
# print(f"Xóa {n_rev} dòng 'Reviews'")

# # Hàm missing (hỗ trợ cả Series và giá trị scalar)
# def is_missing(s):
#     # Nếu là Series (hoặc Index), trả về Series bool
#     if isinstance(s, (pd.Series,)):
#         return s.isna() | (s.astype(str).str.strip() == "") | (s.astype(str).str.lower().isin(["nan", "null", "-", "reviews"]))
#     # Nếu là giá trị đơn (scalar), trả về bool
#     else:
#         if pd.isna(s):
#             return True
#         st = str(s).strip()
#         if st == "":
#             return True
#         if st.lower() in ["nan", "null", "-", "reviews"]:
#             return True
#         return False

# # Rule mới: original_price thiếu → gán = price
# mask_no_price = is_missing(df['original_price'])
# df.loc[mask_no_price, 'original_price'] = df.loc[mask_no_price, 'price']
# print(f"Gán original_price = price: {mask_no_price.sum()} dòng")

# # discount_percent thiếu → 0%
# mask_no_disc = is_missing(df['discount_percent'])
# df.loc[mask_no_disc, 'discount_percent'] = "0%"
# print(f"Điền discount_percent = 0%: {mask_no_disc.sum()} dòng")

# # Chuẩn hóa (giống trên)
# def extract_number(t): 
#     digits = re.sub(r"[^\d]", "", str(t))
#     return int(digits) if digits else np.nan

# df['original_price'] = df['original_price'].apply(extract_number)
# df['discount_percent'] = df['discount_percent'].astype(str).str.replace(r"[^\d%]", "", regex=True).replace("", "0%")

# def calc_price(r):
#     if pd.isna(r['original_price']) or r['original_price'] == 0: return np.nan
#     if r['discount_percent'] == "0%": return r['original_price']
#     try:
#         disc = int(re.sub(r"[^\d]", "", r['discount_percent'])) / 100
#         return int(r['original_price'] * (1 - disc))
#     except:
#         return r['original_price']
# df['price'] = df.apply(calc_price, axis=1).astype('Int64')

# df['comment_count'] = df['comment_count'].apply(lambda x: 0 if is_missing(x) or str(x).strip() == "Reviews" else extract_number(x)).fillna(0).astype(int)
# df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(0)

# def extract_sold(t):
#     if pd.isna(t): return 0
#     s = re.sub(r"[^0-9KM\.]", "", str(t).upper())
#     m = re.match(r"(\d+)\.(\d)([KM])", s)
#     if m: return int(m.group(1)+m.group(2)) * (100 if m.group(3)=="K" else 100000)
#     m = re.match(r"(\d+)([KM])", s)
#     if m: return int(m.group(1)) * (1000 if m.group(3)=="K" else 1000000)
#     digits = re.sub(r"[^\d]", "", s)
#     return int(digits) if digits else 0
# df['sold'] = df['sold'].apply(extract_sold)

# # XÓA TRÙNG link
# dup = df['link'].duplicated().sum()
# df = df.drop_duplicates(subset='link', keep='first')

# # LOẠI BỎ DÒNG THIẾU category, name (original_price đã được gán nên không xóa)
# mask = is_missing(df['category']) | is_missing(df['name'])
# n_final = mask.sum()
# df = df[~mask].copy()

# # # Xuất
# # df.to_csv("products_details_final_clean.csv", index=False, encoding="utf-8-sig")
# # print(f"ĐÃ LƯU: products_details_an_clean.csv → {len(df)} dòng")
# # print(f"→ Xóa 'Reviews': {n_rev} | Trùng link: {dup} | Thiếu category/name: {n_final}")

In [ ]:
# OUTPUT
import os

final_output = "products_details_final_clean.csv"
is_first_write = not os.path.exists(final_output)

# Ghi file: nếu là lần đầu → có header, nếu không → không có header
df.to_csv(
    final_output,
    mode='a',           # 'a' = append (thêm vào cuối)
    header=is_first_write,  # chỉ ghi header nếu file chưa tồn tại
    index=False,
    encoding="utf-8-sig"
)

if is_first_write:
    print(f"ĐÃ TẠO MỚI & GHI FILE: {final_output}")
    print(f"   → Số dòng ghi: {len(df)}")
else:
    print(f"ĐÃ CHÈN THÊM VÀO FILE: {final_output}")
    print(f"   → Số dòng mới thêm: {len(df)}")
    print(f"   → Tổng dòng trong file hiện tại: {sum(1 for _ in open(final_output, encoding='utf-8-sig')) - 1}")  # trừ header

print("TỔNG KẾT")
print("="*50)
print(f"Ban đầu              : {total_rows} dòng")
print(f"Trùng link           : {dup_before} dòng")
print(f"Thiếu dữ liệu bắt buộc: {n_removed} dòng")